# 🔧 Bitcoin Price Prediction - Feature Engineering
> **Objective:** Create and prepare features for machine learning models that predict Bitcoin prices.

This notebook covers:
- 📥 Data Preparation
- 🏗️ Lag Feature Creation
- 📊 Train-Test Split for Multiple Models
- 🔄 Data Resampling
- 📐 Feature Scaling
- 📦 Data Export for Modeling

## 1. Library Imports

In [ ]:
# -*- coding: utf-8 -*-
# =============================================================================
# Feature Engineering for Bitcoin Price Prediction
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")

## 2. Load and Prepare Data
Loading the cleaned dataset and preparing it for feature engineering.

In [ ]:
# =============================================================================
# Load cleaned data
# =============================================================================
df = pd.read_csv('../BTC-USD-Price-History-2010-2024.csv')

# Clean numeric columns
cols_to_clean = ['Volume', 'Open', 'High', 'Low', 'Close']
for col in cols_to_clean:
    df[col] = (df[col]
               .astype(str)
               .str.replace('$', '', regex=False)
               .str.replace(',', '', regex=False)
               .str.replace(' ', '', regex=False)
               .astype(float))

# Convert Date and set as index
df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)
df = df.sort_values('Date')

# For modeling, we focus on 'High' price as our target
# Drop redundant price columns (Close, Low, Open are highly correlated with High)
df_model = df.drop(['Close', 'Low', 'Open'], axis=1)

print(f"✓ Data loaded and prepared")
print(f"  • Shape: {df_model.shape}")
print(f"  • Features: {list(df_model.columns)}")
display(df_model.head())

## 3. Resample to Daily Frequency
Resampling ensures consistent time intervals for time series modeling.

In [ ]:
# =============================================================================
# Resample to daily frequency
# =============================================================================
ts = df_model['High'].resample('D').mean()

print(f"✓ Daily resampling complete")
print(f"  • Observations: {len(ts)}")
print(f"  • Date range: {ts.index.min().date()} to {ts.index.max().date()}")
print(f"  • Missing values: {ts.isna().sum()}")

# Display sample
display(ts.head(10).to_frame('High_Price'))

## 4. Feature Creation

### 4.1 Lag Features
Lag features use past observations to predict future values. This is the core technique for time series forecasting with regression models.

**Why Lag=1?** 
- Bitcoin prices exhibit strong autocorrelation with the previous day's price
- Additional lags (Lag_7, Lag_30) capture weekly and monthly patterns

In [ ]:
# =============================================================================
# Create lag features for regression models
# =============================================================================

# Create DataFrame for regression
data_for_reg = pd.DataFrame(ts)

# Create multiple lag features
data_for_reg['Lag_1'] = data_for_reg['High'].shift(1)    # Previous day
data_for_reg['Lag_2'] = data_for_reg['High'].shift(2)    # 2 days ago
data_for_reg['Lag_3'] = data_for_reg['High'].shift(3)    # 3 days ago
data_for_reg['Lag_7'] = data_for_reg['High'].shift(7)    # 1 week ago
data_for_reg['Lag_30'] = data_for_reg['High'].shift(30)  # 1 month ago

# Drop rows with NaN values (from shifting)
data_for_reg.dropna(inplace=True)

print(f"✓ Lag features created")
print(f"  • Shape with lag features: {data_for_reg.shape}")
print(f"  • Features: {list(data_for_reg.columns)}")
display(data_for_reg.head(10))

### 4.2 Visualizing Lag Relationships
Understanding how current price relates to previous prices.

In [ ]:
# =============================================================================
# Visualize lag relationships
# =============================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

lags = [('Lag_1', '1 Day Lag'), ('Lag_2', '2 Days Lag'), ('Lag_7', '7 Days Lag'), ('Lag_30', '30 Days Lag')]
positions = [(0, 0), (0, 1), (1, 0), (1, 1)]

for (lag_col, title), pos in zip(lags, positions):
    ax = axes[pos[0], pos[1]]
    ax.scatter(data_for_reg[lag_col], data_for_reg['High'], alpha=0.3, s=10, color='#f04747')
    ax.set_xlabel(f'{title}', fontsize=11)
    ax.set_ylabel('Current High Price', fontsize=11)
    ax.set_title(f'{title} vs Current Price', fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Add correlation coefficient
    corr_val = data_for_reg['High'].corr(data_for_reg[lag_col])
    ax.text(0.05, 0.95, f'Corr: {corr_val:.3f}', transform=ax.transAxes, 
            fontsize=12, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()
print("✓ Lag relationship plots generated")
print("\n--- Correlation with Current Price ---")
correlations = data_for_reg.corr()['High'].sort_values(ascending=False)
print(correlations)

## 5. Train-Test Split

### 5.1 For Regression Models (XGBoost, LightGBM, CatBoost)
We use an 80-20 temporal split to prevent data leakage. Time series requires chronological ordering — we cannot use random shuffle.

In [ ]:
# =============================================================================
# Train-Test Split for Regression Models (Temporal)
# =============================================================================

# Features: Use all lag columns, Target: High price
feature_cols = ['Lag_1', 'Lag_2', 'Lag_3', 'Lag_7', 'Lag_30']
X = data_for_reg[feature_cols]
y = data_for_reg['High']

# Temporal split (80% train, 20% test) - maintaining chronological order
train_size = int(len(X) * 0.8)
X_train, X_test = X.iloc[0:train_size], X.iloc[train_size:len(X)]
y_train, y_test = y.iloc[0:train_size], y.iloc[train_size:len(y)]

print("=" * 70)
print("REGRESSION MODEL DATA SPLIT")
print("=" * 70)
print(f"  • Training set   : {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"  • Testing set    : {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"  • Features       : {feature_cols}")
print(f"  • Date range (train): {X_train.index.min().date()} to {X_train.index.max().date()}")
print(f"  • Date range (test) : {X_test.index.min().date()} to {X_test.index.max().date()}")

print("\n--- Training Set Summary ---")
print(f"  • Target mean: ${y_train.mean():.2f}")
print(f"  • Target std: ${y_train.std():.2f}")
print(f"  • Target range: ${y_train.min():.2f} - ${y_train.max():.2f}")

print("\n--- Testing Set Summary ---")
print(f"  • Target mean: ${y_test.mean():.2f}")
print(f"  • Target std: ${y_test.std():.2f}")
print(f"  • Target range: ${y_test.min():.2f} - ${y_test.max():.2f}")

### 5.2 For ARIMA Model
ARIMA requires a univariate time series split (also 80-20 temporal).

In [ ]:
# =============================================================================
# Train-Test Split for ARIMA Model
# =============================================================================
train_size_arima = int(len(ts) * 0.8)
train_data_arima, test_data_arima = ts[0:train_size_arima], ts[train_size_arima:]

print("=" * 70)
print("ARIMA MODEL DATA SPLIT")
print("=" * 70)
print(f"  • Training set   : {len(train_data_arima)} samples ({len(train_data_arima)/len(ts)*100:.1f}%)")
print(f"  • Testing set    : {len(test_data_arima)} samples ({len(test_data_arima)/len(ts)*100:.1f}%)")
print(f"  • Date range (train): {train_data_arima.index.min().date()} to {train_data_arima.index.max().date()}")
print(f"  • Date range (test) : {test_data_arima.index.min().date()} to {test_data_arima.index.max().date()}")
print(f"\n  • Training mean: ${train_data_arima.mean():.2f}")
print(f"  • Testing mean: ${test_data_arima.mean():.2f}")

## 6. Feature Engineering Summary

| Feature Type | Description | Models Using It |
|-------------|-------------|-----------------|
| **Lag_1** | Previous day's price | Regression Models |
| **Lag_2** | Price 2 days ago | Regression Models |
| **Lag_3** | Price 3 days ago | Regression Models |
| **Lag_7** | Price 1 week ago | Regression Models |
| **Lag_30** | Price 1 month ago | Regression Models |
| **Log + Differenced** | Stationary transformed series | ARIMA |

### Data Shapes Summary
| Dataset | Training | Testing |
|---------|----------|---------|
| **Regression** | X: {X_train.shape}, y: {y_train.shape} | X: {X_test.shape}, y: {y_test.shape} |
| **ARIMA** | {len(train_data_arima)} samples | {len(test_data_arima)} samples |

> **Next Step:** Proceed to **Model Training & Evaluation** notebook to build and evaluate prediction models.